# Logistic Regression
Companion to the logistic regression notes.

## 1. One gradient step by hand, checked

In [ ]:
import numpy as np
sigmoid = lambda z: 1 / (1 + np.exp(-z))
X = np.c_[np.ones(4), [1.0, 2.0, 3.0, 4.0]]
y = np.array([0, 0, 1, 1])
w = np.array([-3.0, 1.0])
p = sigmoid(X @ w)
g = X.T @ (p - y)
print("p:", p.round(4), " gradient:", g.round(4))
w = w - 0.5 * g
print("after one step:", w.round(4))
print("Hessian X^T W X:\n", (X.T * (p * (1 - p))) @ X)

## 2. Odds and log-odds

In [ ]:
for prob in [0.5, 0.8, 0.99]:
    odds = prob / (1 - prob)
    print(f"p={prob}: odds={odds:.2f}  log-odds={np.log(odds):.3f}")

## 3. Perfectly separable data: the weights never settle

In [ ]:
from sklearn.linear_model import LogisticRegression
x_sep = np.array([[1.0], [2.0], [3.0], [4.0]])
y_sep = np.array([0, 0, 1, 1])
for C in [1, 100, 1e4, 1e6]:
    m = LogisticRegression(C=C, max_iter=10000).fit(x_sep, y_sep)
    print(f"C={C:>9}: weight {m.coef_[0][0]:8.3f}   p(x=3) = {m.predict_proba([[3.0]])[0, 1]:.6f}")

## 4. Balanced class weights

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
y_small = np.array([0, 0, 0, 0, 1, 0, 1, 0, 0, 0])
print(compute_class_weight("balanced", classes=np.array([0, 1]), y=y_small))

## 5. The bank data: tune penalty, C and class weights

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

bd = pd.read_csv("../../data/bd_train.csv")
bd = bd.drop(columns=["REF_NO", "post_code", "post_area"])        # IDs / too many unique values
y = (bd.pop("Revenue.Grid") == 1).astype(int)                      # 1 = high-revenue customer
X = pd.get_dummies(bd, drop_first=True, dtype=float)
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)
print(x_train.shape, round(y_train.mean(), 3))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([("scale", StandardScaler()),
                 ("logit", LogisticRegression(solver="liblinear", max_iter=2000))])
params = {"logit__penalty": ["l1", "l2"],
          "logit__C": [0.001, 0.01, 0.1, 1, 10],
          "logit__class_weight": [None, "balanced"]}
gs = GridSearchCV(pipe, params, cv=5, scoring="roc_auc")
gs.fit(x_train, y_train)
print("best:", gs.best_params_, " CV AUC:", round(gs.best_score_, 4))

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

model = gs.best_estimator_
p_test = model.predict_proba(x_test)[:, 1]
print("test AUC:", round(roc_auc_score(y_test, p_test), 4))
fpr, tpr, thr = roc_curve(y_test, p_test)
k = np.argmax(tpr - fpr)
print(f"KS = {tpr[k] - fpr[k]:.3f} at cutoff {thr[k]:.3f}")
print("non-zero weights:", int((model[-1].coef_ != 0).sum()), "of", x_train.shape[1])

## 6. The largest weights as odds ratios

In [ ]:
w = pd.Series(model[-1].coef_[0], index=x_train.columns)
top = w.abs().sort_values(ascending=False).index[:6]
print(pd.DataFrame({"weight": w[top], "odds ratio per 1 sd": np.exp(w[top])}).round(3))

## 7. A two-feature decision boundary

In [ ]:
import matplotlib.pyplot as plt
bd2 = pd.read_csv("../../data/bd_train.csv")
y2 = (bd2["Revenue.Grid"] == 1).astype(int)
X2 = np.c_[np.log1p(bd2["Online.Purchase.Amount"]), np.log1p(bd2["Life.Insurance"])]
m2 = LogisticRegression().fit(X2, y2)
w0, (w1, w2) = m2.intercept_[0], m2.coef_[0]
plt.scatter(X2[:, 0], X2[:, 1], c=y2, s=4, cmap="coolwarm", alpha=0.5)
xs = np.linspace(0, 8.5, 100)
for c, ls in [(0.5, "-"), (0.2, "--")]:
    plt.plot(xs, (np.log(c / (1 - c)) - w0 - w1 * xs) / w2, "k", ls=ls, label=f"cutoff {c}")
plt.ylim(-0.3, 8.2); plt.legend(); plt.xlabel("log(1+online purchases)"); plt.ylabel("log(1+life insurance)")

## Try it yourself
1. Second gradient step in section 1.
2. Balanced weights for 30 ones and 970 zeros.
3. `np.log1p` on the spending columns (careful with negative `Portfolio.Balance`): does test AUC change?
4. Refit without `class_weight`: compare test AUC and the KS cutoff.

## Undersampling vs class weights

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score

bd = pd.read_csv("../../data/bd_train.csv").drop(columns=["REF_NO", "post_code", "post_area"])
y = (bd.pop("Revenue.Grid") == 1).astype(int)
X = pd.get_dummies(bd, drop_first=True, dtype=float)
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)

# undersampling: keep every minority row, and an equal-sized random sample of the majority (training data only)
rng = np.random.default_rng(0)
minority = np.where(y_train == 1)[0]
majority = rng.choice(np.where(y_train == 0)[0], size=len(minority), replace=False)
keep = np.concatenate([minority, majority])
x_under, y_under = x_train.iloc[keep], y_train.iloc[keep]
print("training rows: all", len(y_train), "| undersampled", len(y_under), "| positive share", round(y_under.mean(), 2))

setups = {"no adjustment": (x_train, y_train, None), "class_weight='balanced'": (x_train, y_train, "balanced"),
          "undersampled 1:1": (x_under, y_under, None)}
for name, (xt, yt, cw) in setups.items():
    model = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight=cw, max_iter=5000)).fit(xt, yt)
    p = model.predict_proba(x_test)[:, 1]
    pred = (p >= 0.5).astype(int)
    print(f"{name:24s}: AUC {roc_auc_score(y_test, p):.4f}  mean p {p.mean():.3f}  "
          f"at cutoff 0.5: precision {precision_score(y_test, pred):.3f} recall {recall_score(y_test, pred):.3f}")

Correcting the probabilities of an undersampled model:

In [ ]:
def correct_undersampled(p, beta):
    """beta = share of the majority class that was kept."""
    return beta * p / (beta * p - p + 1)

beta = len(majority) / (y_train == 0).sum()
model = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000)).fit(x_under, y_under)
p = model.predict_proba(x_test)[:, 1]
print(f"kept {beta:.3f} of the majority; mean p before {p.mean():.3f}, after correction {correct_undersampled(p, beta).mean():.3f},"
      f" actual rate {y_test.mean():.3f}")